### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="video_transcoding_time_prediction",
    dataset_year="2015",
    domain_str="technology & internet",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C58C9K",
    download_description="""
wget https://archive.ics.uci.edu/static/public/335/online+video+characteristics+and+transcoding+time+dataset.zip \
&& unzip online+video+characteristics+and+transcoding+time+dataset.zip transcoding_mesurment.tsv \
&& rm online+video+characteristics+and+transcoding+time+dataset.zip \
&& mkdir -p local-data-warehouse/video_transcoding_time_prediction \
&& mv transcoding_mesurment.tsv local-data-warehouse/video_transcoding_time_prediction/
""",
    # References
    academic_reference_bibtex="""@inproceedings{deneke2014video,
  title={Video transcoding time prediction for proactive load balancing},
  author={Deneke, Tewodors and Haile, Habtegebreil and Lafond, S{\'e}bastien and Lilius, Johan},
  booktitle={2014 IEEE International Conference on Multimedia and Expo (ICME)},
  pages={1--6},
  year={2014},
  organization={IEEE}
}
""",
    academic_reference_bibtex_key="deneke2014video",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
We use the tabular prediction task from the UCI archive, which is a video transcoding time prediction task. The data is non-IID and grouped by video, with multiple transcoding measurements per video. The target variable is the transcoding time, and the features include various characteristics of the videos and transcoding settings.

- Some features on the videos (like URL and category) are not part of this dataset as we got it from UCI, but this sounds like a good idea for the task.
- The data includes several transcoding per video (also with different output target formats), which creates a non-IID setting. We will use the video id as the group label, and we will make sure to split the data such that all transcoding measurements for a given video are in the same split (train/test).
- We log scale the target variable.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="log_transcoding_time",
    problem_type="regression",
    objective_metric_name="rmse",
    # For grouped data
    group_on="video_id",
    group_labels="per_sample",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "transcoding_mesurment.tsv", sep="\t")
print("Loaded data shape:", df.shape)

df = df.drop(columns=[
    # Target colum / leaking
    "umem",
    # Constant
    "b_size",
])
df = df.rename(columns={
    "utime": "log_transcoding_time",
    "id": "video_id",
})

as_cat_dtype = [
    "video_id",
    "codec",
    "o_codec",
]
df[as_cat_dtype] = df[as_cat_dtype].astype("category")
df["log_transcoding_time"] = np.log(df["log_transcoding_time"])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For Grouped Non-IID data
split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
    random_state=split_random_state,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()